# B7 — Full-layout stitching and exact-coordinate recovery

Use a GPU runtime. This notebook scans every non-overlapping B6 central output on the three validation and three development-confirmation layout families, including original source variants. It selects one deployment policy on validation only, locally recovers exact KLayout `m1.2` edge pairs, and keeps the B9 final holdout unopened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
PERSISTENT_ROOT = DRIVE_ROOT / 'ADVLSI2_B7'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_ROOT

In [ ]:
import subprocess, sys

REPO = Path('/content/ADVLSI2_Project_updated')
BRANCH = 'agent/b7-full-layout-stitching'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/nocleo/ADVLSI2_Project_updated.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_b7_full_layout.py', '-v'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

CHECKPOINT_ROOT = DRIVE_ROOT / 'ADVLSI2_B6_2/b6_multitask_unet'
required = [CHECKPOINT_ROOT / f'seed_{seed}/best.pth' for seed in (42, 43, 44)]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing accepted B6.2 checkpoints: ' + ', '.join(missing)
required

In [ ]:
OUTPUT_DIR = PERSISTENT_ROOT / 'b7_full_layout'
command = [
    sys.executable, 'scripts/run_b7_full_layout.py',
    '--checkpoint-dir', str(CHECKPOINT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cuda',
    '--batch-size', '32',
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import json
from IPython.display import Markdown, display

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
assert summary['status'] == 'complete'
assert summary['official_result'] is True
assert summary['untouched_b9_final_holdout_used'] is False
display(Markdown((OUTPUT_DIR / 'README.md').read_text()))
summary['acceptance']

In [ ]:
import zipfile
from google.colab import files

archive = Path('/content/ADVLSI2_B7_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(item for item in OUTPUT_DIR.rglob('*') if item.is_file()):
        relative = path.relative_to(OUTPUT_DIR)
        if relative.parts[0] in {'layout_cache', 'scan_cache'}:
            continue
        output.write(path, relative.as_posix())
print(f'Result archive: {archive.stat().st_size / 1e6:.1f} MB')
files.download(str(archive))